# Rich Chess Ebook — extraction pipeline

Turns **PDF chess books** into `.rce` archives the Flutter app can read, several in
one pass so their numbers can be read side by side.

Two notations are read straight from the text layer: figurine Unicode, and plain
letters in `en`, `fr`, `de`, `es`, `it` or `nl`.

A book whose piece symbols are **drawn** — a scan, or a figurine font — holds no
readable symbol in that layer at all. Step 4 reports those as
`needs_glyph_recovery = True`, and a trained classifier reads their symbols off the
page images and writes them back in as figurines.

**The classifier is handed to every book, once, in step 3.** `run()` engages it only
on the books whose detection asked for it, so there is no per-book flag to set and no
way to lose a scanned book by forgetting one — which is the mistake this notebook was
rewritten to make impossible.

The logic is not in these cells but in the repository's `rce_pipeline` package: a
notebook is neither testable nor readable as a diff, whereas each step here is a
module that can be fixed and re-run on its own.

| Step | Module | Artefact written |
| --- | --- | --- |
| 1. Text + per-character geometry | `extract.py` | `work/<book>/01_pages.json` |
| 1c. Piece symbols read off the image | `scan.py`, `glyphs.py` | `work/<book>/01b_glyphs.json` |
| 2. Notation detection | `notation.py` | `work/<book>/02_notation.json` |
| 3a. Tokenising | `tokenize.py` | `work/<book>/03_tokens.json` |
| 3b + 4. Move tree, legality, FEN | `parse.py` | `work/<book>/04_moves.json` |
| 5. Packaging | `package.py` | `<book>.rce` |

Each step reads the previous one's artefact, so editing `parse.py` and re-running does
not redo extraction — by far the slowest part.


In [1]:
!rm -rf RichChessEbooks

## 1. Install


In [2]:
!pip install -q pymupdf chess
# Only needed for a scanned or figurine-font book (step 4b):
!pip install -q scikit-learn scikit-image pillow


## 2. Get the pipeline code

Set `REPO_URL` to clone from GitHub. Leave it at `None` and the cell asks you to
upload a ZIP of the `pipeline/` directory instead.


In [3]:
REPO_URL = "https://github.com/loloof64/RichChessEbooks.git"  # or None to upload a ZIP

import os, sys, zipfile

if REPO_URL:
    if not os.path.isdir("RichChessEbooks"):
        !git clone --depth 1 $REPO_URL
    PIPELINE_DIR = "RichChessEbooks/pipeline"
elif os.path.isdir("pipeline"):
    PIPELINE_DIR = "pipeline"
else:
    from google.colab import files
    print("Upload a ZIP containing the pipeline/ directory")
    for name in files.upload():
        with zipfile.ZipFile(name) as archive:
            archive.extractall(".")
    PIPELINE_DIR = "pipeline"

sys.path.insert(0, os.path.abspath(PIPELINE_DIR))

import rce_pipeline
from rce_pipeline import extract, notation, tokenize, parse, package, pipeline
print("rce_pipeline", rce_pipeline.__version__, "from", PIPELINE_DIR)


Cloning into 'RichChessEbooks'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 227 (delta 19), reused 220 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (227/227), 374.44 KiB | 4.11 MiB/s, done.
Resolving deltas: 100% (19/19), done.
rce_pipeline 0.1.0 from RichChessEbooks/pipeline


## 3. The books, and the classifier

`BOOKS` maps a short label to a PDF path — the label is what names the reports and
the `.rce` files below. Reading from Drive is worth it for anything large; leave the
dict empty to upload instead.

`GLYPH_MODEL` is the trained piece classifier, and it is set **once, here, for all
the books**. Passing it to a book that does not need it costs nothing: `run()` checks
each book's own `needs_glyph_recovery` before engaging it.

**Uploads happen once.** Whatever you upload is written into `/content/books` and
`/content/model` and picked up again on every later run of this cell, so re-running it
after an error does not ask for the 48 MB classifier a second time. Set `REUPLOAD =
True` to replace what is there. Files already sitting in `/content` — from an earlier
upload in this session — are adopted rather than asked for again.

For anything large, mounting Drive still beats uploading:

```python
from google.colab import drive; drive.mount("/content/drive")
```

then point `BOOKS` and `GLYPH_MODEL` straight at `/content/drive/MyDrive/...`.


In [ ]:
from google.colab import drive; drive.mount("/content/drive")

# label -> path. Leave empty to upload.
BOOKS = {
    "SuperAttaquant": "/content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/CommentDevenirSuperAttaquant.pdf",
    "BoussoleSurEchiquier": "/content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/BoussoleSurEchiquier.pdf",
    "ChessStrategy_Grivas_1": "/content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/ChessTrategy_Grivas_1.pdf",
    "Tactics": "/content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_texte/TacticsForTournamentPlayer.pdf",
}

GLYPH_MODEL = "/content/drive/MyDrive/entrainement_ocr_echecs/chess_glyphs_classifier.zip"

BOOKS_DIR = "/content/books"
MODEL_DIR = "/content/model"
REUPLOAD = False  # True to ignore what is already there and upload again

import glob, os, re


def _save(target_dir):
    """Prompt for files and write them into `target_dir` under their own names.

    `files.upload()` hands back `{name: bytes}`, so the bytes are written here
    rather than left where Colab dropped them. Colab saves a second upload of
    the same book as `book (1).pdf`, which would then be read as a second book.
    """
    from google.colab import files

    os.makedirs(target_dir, exist_ok=True)
    for name, data in files.upload().items():
        with open(os.path.join(target_dir, name), "wb") as handle:
            handle.write(data)


def _discover(*directories, pattern):
    """Files matching `pattern`, first directory winning, `name (1)` folded away.

    Sorted shortest basename first so that `book.pdf` beats the `book (1).pdf`
    Colab wrote beside it: plain alphabetical order puts the duplicate first,
    the space sorting before the dot.
    """
    found = {}
    for directory in directories:
        paths = glob.glob(os.path.join(directory, pattern))
        for path in sorted(paths, key=lambda p: (len(os.path.basename(p)), p)):
            stem = os.path.splitext(os.path.basename(path))[0]
            found.setdefault(re.sub(r" \(\d+\)$", "", stem), path)
    return found


if not BOOKS:
    if REUPLOAD or not _discover(BOOKS_DIR, "/content", pattern="*.pdf"):
        print("Upload one or more PDFs")
        _save(BOOKS_DIR)
    BOOKS = _discover(BOOKS_DIR, "/content", pattern="*.pdf")

if not BOOKS:
    raise SystemExit("no PDF found — re-run this cell and upload at least one")

if GLYPH_MODEL is None:
    # A pipeline ZIP may also be sitting in /content, so the classifier is looked
    # for by name before falling back to any archive at all.
    def _models():
        return _discover(MODEL_DIR, "/content", pattern="*classifier*.zip") or _discover(
            MODEL_DIR, "/content", pattern="*.zip"
        )

    if REUPLOAD or not _models():
        print("Upload chess_glyphs_classifier.zip")
        _save(MODEL_DIR)
    GLYPH_MODEL = next(iter(_models().values()), None)

width = max(len(label) for label in BOOKS)
for label, path in BOOKS.items():
    print(f"{label:<{width}} {extract.page_count(path):>5} pages  {path}")
print(f"\nclassifier: {GLYPH_MODEL}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SuperAttaquant            12 pages  /content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/CommentDevenirSuperAttaquant.pdf
BoussoleSurEchiquier      13 pages  /content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/BoussoleSurEchiquier.pdf
ChessStrategy_Grivas_1    10 pages  /content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/ChessTrategy_Grivas_1.pdf

classifier: /content/drive/MyDrive/entrainement_ocr_echecs/chess_glyphs_classifier.zip


## 4. What each book is

Detection runs on a sample of each book. The result is a **conclusion to confirm**,
not a question asked cold: the counts supporting it are printed underneath.

The column that decides the rest of the run is the last one.

- `style` — `figurine_unicode` and `letters` are read straight from the text layer.
- `neutral` — moves naming no piece at all (pawn moves, castlings). Present in every
  language, so it proves "this is chess notation" without favouring one.
- `glyph recovery` — `REQUIRED` means the piece symbols are drawn on the page and
  the text layer never held them. Three kinds of book land here: a scan, a figurine
  font caught by its font name, and the one that fools every other test — a book
  reported as `letters` at 0% confidence with every language scoring zero, whose
  fonts look ordinary (`Helvetica`) and whose only moves are the neutral ones. A
  book of nothing but pawn moves does not exist, so that combination *is* the
  signature of drawn symbols.


In [ ]:
SAMPLE_FIRST_PAGE = 1
SAMPLE_LAST_PAGE = 40

samples, reports = {}, {}
for label, path in BOOKS.items():
    samples[label] = extract.extract_pages(
        path, first_page=SAMPLE_FIRST_PAGE, last_page=SAMPLE_LAST_PAGE
    )
    reports[label] = notation.detect_notation(samples[label])

width = max(len(label) for label in BOOKS)
header = f"{'book':<{width}} {'style':<17} {'lang':<5} {'conf':>5} {'neutral':>8}  glyph recovery"
print(header)
print("-" * len(header))
for label, report in reports.items():
    print(
        f"{label:<{width}} {report.style:<17} {str(report.language or '-'):<5} "
        f"{report.confidence:>4.0%} {report.neutral_move_count:>8}  "
        f"{'REQUIRED' if report.needs_glyph_recovery else 'not needed'}"
    )


In [ ]:
# The evidence behind each verdict, book by book.
for label, report in reports.items():
    print(f"\n{'=' * 72}\n{label}\n{'=' * 72}")
    print(report.summary())
    print("\nMost frequent fonts:")
    for name, count in list(extract.font_inventory(samples[label]).items())[:5]:
        print(f"  {count:>7}  {name}")


### Read one book's layer with your own eyes

`SELECTED` names the book every single-book check below works on — the previews in
4b, the boxes in step 7, the move tree in step 8. Change it and re-run those cells to
move to another book.

If the moves below show up as garbage — `4)xf7`, `We6`, `2xf7` — the layer is OCR
output. Its prose is usually fine and its move numbers and squares mostly right; only
the piece symbols are hopeless, which is what step 4b is for. Prose that reads cleanly
while moves do not is the signature.


In [ ]:
SELECTED = next(iter(BOOKS))

page = samples[SELECTED][len(samples[SELECTED]) // 2]
print(f"--- {SELECTED}, page {page.number} ({page.width} x {page.height} pt) ---")
print(page.text[:1200])


## 4b. Preview the recovered symbols

Skip this if no book was marked `REQUIRED` above; step 5 applies the classifier on
its own either way. This section exists to look at what it does before trusting it,
on `SELECTED`.

The classifier is a random forest over five classes — King, Queen, Rook, Bishop,
Knight — trained outside this repository. It has no "not a piece" class, so a crop is
only shown to it when its shape already says piece: about twice as wide as the page's
letters, and nearly square. What comes back above `min_confidence` is written into the
pages as a figurine, carrying the box of the printed symbol rather than of whatever
the scanner read there.

Measured on two hand-read pages of a French scanned book, 54 symbols printed: **53
recovered, none invented**, at the default confidence of 0.45. Lower it and phantom
pieces appear in the prose; raise it and bishops start dropping out first.


In [ ]:
PREVIEW_PAGES = (SAMPLE_FIRST_PAGE, SAMPLE_FIRST_PAGE + 1)
MIN_CONFIDENCE = None  # None uses glyphs.DEFAULT_MIN_CONFIDENCE

from rce_pipeline import glyphs, scan

classifier = glyphs.GlyphClassifier.load(GLYPH_MODEL)
min_confidence = glyphs.DEFAULT_MIN_CONFIDENCE if MIN_CONFIDENCE is None else MIN_CONFIDENCE

preview = extract.extract_pages(
    BOOKS[SELECTED], first_page=PREVIEW_PAGES[0], last_page=PREVIEW_PAGES[1]
)
before = {
    p.number: [line.text for line in scan.notation_lines(scan.segment_lines(p))]
    for p in preview
}
repaired, found = glyphs.recover_pieces(
    BOOKS[SELECTED], preview, classifier, min_confidence=min_confidence
)

placed, total = glyphs.placement_score(repaired)
if total:
    print(f"{total} symbols recovered, {placed} spliced into a move ({placed / total:.0%})\n")
else:
    print("no symbols found — check MIN_CONFIDENCE, and that these pages carry moves\n")
for page in repaired:
    after = [line.text for line in scan.notation_lines(scan.segment_lines(page))]
    for old, new in zip(before[page.number], after):
        if old != new:
            print(f"  before: {old}\n  after : {new}\n")


A low "spliced into a move" share means the symbols were recognised and then written
into the wrong characters. That is not the classifier: it is the text layer's boxes.
Tesseract divides a word's box evenly among the characters it read, so a layer that
read `tZJg3` where `♘g3` is printed puts every box in the wrong place, and the symbol
lands beside the move instead of at its head. Around 90% is a well-boxed layer; below
60% the moves from this book are not worth parsing, and the page images would have to
be read in full rather than repaired.


In [ ]:
# The crops the classifier was shown, with what it made of them.
from PIL import Image
import io

page = repaired[0]
source = next(p for p in preview if p.number == page.number)
lines = scan.notation_lines(scan.segment_lines(source))
with scan.PageRenderer(BOOKS[SELECTED]) as renderer:
    for line in lines[:4]:
        image = renderer.crop(line)
        display(Image.open(io.BytesIO(image.png)))
        on_line = [
            g for g in found
            if g.page == page.number
            and line.bbox.y <= g.bbox.y + g.bbox.h / 2 <= line.bbox.y + line.bbox.h
        ]
        print("  ".join(
            f"{g.figurine} {g.confidence:.2f}"
            for g in sorted(on_line, key=lambda g: g.bbox.x)
        ) or "(nothing)")


## 5. Full pipeline, on every book

`FIRST_PAGE` / `LAST_PAGE` restrict the work to part of each book — start small, on a
chapter whose content you know, before launching 400 pages.

- `strict_numbering=True` only reads a move when a move number has just announced one,
  or when variation brackets make the context unambiguous. That is what separates `Bb5`
  from a figure caption reading "diagram b4". Set it to `False` for a book that prints
  long unnumbered sequences.
- `FORCE_LANGUAGE` is worth setting **for any letters book you know**. It decides which
  alphabet piece initials come from, and the alphabets overlap: `R` is the King in
  French and the Rook in English. Both readings are frequently legal in the same
  position, so a wrong language does not fail — it produces a different game. Figurine
  books do not need it: their symbols map to SAN letters directly.
- `glyph_model` is passed for every book and engaged only where step 4 said `REQUIRED`.

| Language | King | Queen | Rook | Bishop | Knight |
| --- | --- | --- | --- | --- | --- |
| `en` | K | Q | R | B | N |
| `fr` | R | D | T | F | C |
| `de` | K | D | T | L | S |
| `es` / `it` | R | D | T | A | C |
| `nl` | K | D | T | L | P |

One book failing does not stop the others: its traceback is printed and the loop moves
on.


In [ ]:
FIRST_PAGE = 1
LAST_PAGE = 40
STRICT_NUMBERING = True
FORCE_LANGUAGE = {}       # label -> "fr" / "en" / "de" / "es" / "it" / "nl"
FORCE_NOTATION = {}       # label -> "figurine_unicode" / "letters", to bypass step 2

import traceback

results = {}
for label, path in BOOKS.items():
    print(f"\n{'=' * 72}\n{label}\n{'=' * 72}")
    try:
        results[label] = pipeline.run(
            path,
            work_dir=f"/content/work/{label}",
            output_path=f"/content/{label}.rce",
            first_page=FIRST_PAGE,
            last_page=LAST_PAGE,
            strict_numbering=STRICT_NUMBERING,
            force_notation=FORCE_NOTATION.get(label),
            force_language=FORCE_LANGUAGE.get(label),
            glyph_model=GLYPH_MODEL,  # engaged only where step 4 said REQUIRED
        )
        print(results[label].report())
    except Exception:
        traceback.print_exc()


## 6. The measurement

### Check the parse is healthy first

The pipeline starts every game from the standard initial position. On a page whose
game begins at move 23, every move is played on a board unrelated to the book,
everything comes out `broken`, and the ambiguity counts below are noise.

So read `ok` first. If it is near zero, the page range is wrong — change it until a
game starts at `1.` **Do not interpret the ambiguity columns until `ok` looks sane.**

### Then read the ambiguity columns

An ambiguity is a move naming a square two pieces reach — `Nd2` with knights on b1 and
f3 — where the book printed a disambiguating letter (`Nbd2`) the pipeline did not see.
`python-chess` already excludes a pinned piece's moves from `legal_moves`, so the usual
reason a book omits the letter never produces a false ambiguity here. That is what
makes the signal clean, and worth measuring rather than guessing at.

| Column | Meaning |
| --- | --- |
| `ambig` | total ambiguous moves |
| `repair` | **the number that answers the question** — cases sitting below a move accepted after an OCR repair, so the board may already be wrong |
| `clean` | cases on lines with no repair above them: the board is sound and the token itself lost the letter |

| Result | Conclusion | Next |
| --- | --- | --- |
| `repair` dominates | the ambiguity is a symptom, not the disease | tighten `_MAX_REPAIR_COST`, and **do not** build the lookahead |
| `clean` dominates | the board is sound, the token lost the letter | build the lookahead |
| `ambig` near zero on every book | the problem is theoretical for this corpus | surfacing the candidates to the app is enough |

`nearest_repair_plies` gives the distance from each case to the repair above it: a 1
or 2 damns that repair, a 9 is more likely coincidence.


In [ ]:
width = max(len(label) for label in results) if results else 4
header = (
    f"{'book':<{width}} {'moves':>6} {'ok':>6} {'unc':>5} {'brk':>5}   "
    f"{'ambig':>5} {'repair':>7} {'clean':>6} {'figurine':>9}"
)
print(header)
print("-" * len(header))
for label, result in results.items():
    counts = result.parsed.counts()
    diagnosis = result.parsed.ambiguity_diagnosis()
    print(
        f"{label:<{width}} {counts['moves']:>6} {counts['ok']:>6} {counts['uncertain']:>5} "
        f"{counts['broken']:>5}   {diagnosis['total']:>5} "
        f"{diagnosis['downstream_of_repair']:>7} {diagnosis['clean_line']:>6} "
        f"{diagnosis['settled_from_consumed']:>9}"
    )

for label, result in results.items():
    print(f"\n{label}: {result.parsed.ambiguity_diagnosis()}")


## 7. What did not get through

`broken` first: no legal reading was found, so the move is a hole in the line. Then
`uncertain`, accepted after repairing a look-alike scanning error (`0`/`O`, `1`/`l`,
`8`/`B`) — the `repair` field says what was substituted.

Repairs are deliberately conservative. Allowing one arbitrary wrong character would
recover more moves and would also turn `Qh9` into `Qh5` and `Nc6` into `Nc3`: squares
differ by a single character all the time, so the pipeline would emit legal but wrong
moves that silently corrupt every position further down the line.

These moves keep their page and their box, so they stay clickable in the app — which
is where they are meant to be corrected.


In [ ]:
result = results[SELECTED]

for move in result.problems(limit=25):
    detail = move.repair["reason"] if move.repair else ""
    print(f"[{move.status:>9}] p.{move.page:>3}  {move.san:<8} conf={move.confidence:.2f}  {detail}")

print(f"\n{len(result.parsed.skipped)} tokens dropped before validation:")
for skipped in result.parsed.skipped[:15]:
    print(f"  p.{skipped['page']:>3}  {skipped['text']:<10} {skipped['reason']}")


## 8. Check the boxes by eye

This is the most useful check in the notebook. A box off by a few points shows up in no
counter, but makes the clickable zone useless in the app. So render the page and draw
the boxes on top of it.

The code converts `.rce` coordinates (origin bottom-left) back to MuPDF's (origin
top-left) — the same round trip Flutter makes, in reverse. If the frames land on the
moves, the convention is right on both sides.


In [ ]:
try:
    import pymupdf as fitz
except ImportError:
    import fitz
from PIL import Image, ImageDraw

result = results[SELECTED]
PREVIEW_PAGE = result.parsed.moves[0].page if result.parsed.moves else FIRST_PAGE
ZOOM = 2.0
STATUS_COLOURS = {"ok": (0, 160, 0), "uncertain": (220, 140, 0), "broken": (210, 0, 0)}

doc = fitz.open(BOOKS[SELECTED])
page = doc[PREVIEW_PAGE - 1]
pixmap = page.get_pixmap(matrix=fitz.Matrix(ZOOM, ZOOM))
image = Image.frombytes("RGB", (pixmap.width, pixmap.height), pixmap.samples)
draw = ImageDraw.Draw(image)

page_height = page.rect.height
drawn = 0
for move in result.parsed.moves:
    if move.page != PREVIEW_PAGE:
        continue
    b = move.bbox
    top = page_height - b.y - b.h  # flip back to MuPDF's top-left origin
    draw.rectangle(
        [b.x * ZOOM, top * ZOOM, (b.x + b.w) * ZOOM, (top + b.h) * ZOOM],
        outline=STATUS_COLOURS[move.status],
        width=2,
    )
    drawn += 1

doc.close()
print(f"{SELECTED}: {drawn} boxes drawn on page {PREVIEW_PAGE}")
image


## 9. Check the move tree

Variations are reconstructed from `parent_id`, never from array order. This display
follows those links, which checks along the way that they are coherent.


In [ ]:
from collections import defaultdict

GAME_INDEX = 0     # which game to show
MAX_LINES = 80

result = results[SELECTED]
game = result.parsed.games[GAME_INDEX]
children = defaultdict(list)
for move in result.parsed.moves:
    if move.game_id == game.id:
        children[move.parent_id].append(move)
for siblings in children.values():
    siblings.sort(key=lambda m: m.variation_index)

printed = 0

def show(move_id, depth):
    global printed
    for child in children[move_id]:
        if printed >= MAX_LINES:
            return
        printed += 1
        number = f"{(child.ply + 1) // 2}." + ("" if child.ply % 2 else "..")
        mark = {"ok": " ", "uncertain": "~", "broken": "!"}[child.status]
        note = f"    [{child.comment[:60]}]" if child.comment else ""
        print("  " * depth + f"{mark} {number}{child.san}{note}")
        # Only a variation shifts the indentation; the main line stays flush.
        show(child.id, depth + 1 if child.variation_index else depth)

title = game.title or "(untitled)"
print(f"=== {SELECTED} / {game.id} - {title} (p.{game.page_start}) ===")
show(None, 0)


## 10. Download the archives

Each `.rce` holds one original PDF **unchanged**, plus `manifest.json` and
`moves.json`. These are the files the Flutter app imports.


In [ ]:
import zipfile
from google.colab import files

for label, result in results.items():
    if not result.rce_path:
        continue
    print(f"\n{label} — {result.rce_path}")
    with zipfile.ZipFile(result.rce_path) as archive:
        for info in archive.infolist():
            print(f"  {info.file_size:>12,} B  {info.filename}")

for result in results.values():
    if result.rce_path:
        files.download(result.rce_path)
